# 01 - Train the Segmentation Head (SIIM-ACR Pneumothorax)

Trains DenseNet-121 encoder + segmentation decoder (Tiramisu-style, with skip connections) end-to-end, using the Dice loss. Encoder starts from ImageNet weights.

In [ ]:
import os
import sys

REPO_NAME = "multi-task-chest-xray-analysis-system"
PROJECT_DIR = f"/kaggle/working/{REPO_NAME}"

assert os.path.exists(PROJECT_DIR), f"{PROJECT_DIR} does not exist!"

sys.path.insert(0, PROJECT_DIR)

print("Project directory:", PROJECT_DIR)

import torch
from src import config, engine
from src.models.mtl_model import MultiCheXNet
from src.datasets.siim_acr import (SIIMACRDataset, build_siim_dataframe,
                                    get_default_train_augmentation, train_val_split)
from src.utils.visualize import show_image_mask, show_training_curves

print("device:", config.DEVICE)


## 1. Point these at your downloaded data (see notebook 00)

In [ ]:
SEG_CSV_PATH = "/kaggle/input/datasets/jesperdramsch/siim-acr-pneumothorax-segmentation-data/train-rle.csv"
SEG_IMG_PATH = "/kaggle/input/datasets/jesperdramsch/siim-acr-pneumothorax-segmentation-data"
DET_CSV_PATH = "/kaggle/input/competitions/rsna-pneumonia-detection-challenge/stage_2_train_labels.csv"
DET_IMG_PATH = "/kaggle/input/competitions/rsna-pneumonia-detection-challenge/stage_2_train_images"


df = build_siim_dataframe(SEG_CSV_PATH, SEG_IMG_PATH)
print(len(df), "rows,", df['ImageId'].nunique(), "unique images")
df.head()


In [ ]:
from src.datasets.siim_acr import compute_pos_neg_sample_weights

train_ids, val_ids = train_val_split(df, val_fraction=0.15)
print(len(train_ids), "train images,", len(val_ids), "val images")

train_ds = SIIMACRDataset(df, train_ids, augmentation=get_default_train_augmentation())
val_ds = SIIMACRDataset(df, val_ids, augmentation=None)

# Only ~20% of images actually contain a pneumothorax mask. Plain shuffling
# lets the model converge to "predict nothing" within ~1 epoch (see notebook
# discussion). A WeightedRandomSampler balances positive/negative images in
# every batch instead of relying on class weighting inside the loss alone.
train_weights = compute_pos_neg_sample_weights(df, train_ids)
train_sampler = torch.utils.data.WeightedRandomSampler(
    train_weights, num_samples=len(train_weights), replacement=True)

BATCH_SIZE = 16   # T4 16GB handles this comfortably at 256x256
from torch.utils.data import DataLoader
# num_workers=0 avoids the "AssertionError: can only test a child process"
# noise from DataLoader worker cleanup under Kaggle/Jupyter's forked
# multiprocessing + CUDA combination. It's purely cosmetic, but at this
# image size/batch size the throughput cost is negligible.
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=train_sampler,
                          num_workers=0, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)


## 2. Peek at a training sample

In [ ]:
img, mask, label = train_ds[0]
show_image_mask(img.permute(1,2,0).numpy(), mask[0].numpy(), title=f"class={label}")


## 3. Build the model (segmentation head only) and train

In [ ]:
model = MultiCheXNet(pretrained_encoder=True, add_classifier=False,
                     add_detector=False, add_segmenter=True).to(config.DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
scaler = torch.amp.GradScaler('cuda', enabled=(config.DEVICE == "cuda"))
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=3)

EPOCHS = 25
history = engine.fit(
    engine.train_one_epoch_segmenter, engine.evaluate_segmenter,
    model, train_loader, val_loader, optimizer, EPOCHS,
    device=config.DEVICE, checkpoint_path="/kaggle/working/multi-task-chest-xray-analysis-system/segmenter_best.pt",
    # pos_dice = dice averaged only over images that actually contain
    # pneumothorax; the plain "dice" metric is dominated by trivially-easy
    # empty-mask images and hides whether the model is really learning.
    monitor="pos_dice", mode="max", scaler=scaler,
    scheduler=scheduler, patience=7, pos_weight=4.0,
)


## 4. Training curves & qualitative results

In [ ]:
show_training_curves({k: v for k, v in history.items() if 'loss' in k})
show_training_curves({k: v for k, v in history.items() if k in ('train_dice','val_dice')})
show_training_curves({k: v for k, v in history.items() if 'pos_dice' in k})


In [ ]:
model.load("/kaggle/working/multi-task-chest-xray-analysis-system/segmenter_best.pt", map_location=config.DEVICE)
model.eval()

img, mask, label = val_ds[0]
with torch.no_grad():
    pred = model(img.unsqueeze(0).to(config.DEVICE))["seg"][0,0].cpu().numpy()

show_image_mask(img.permute(1,2,0).numpy(), mask[0].numpy(), pred_mask=pred>0.5)


Checkpoint saved to `/content/segmenter_best.pt` - this file only contains the segmentation head + encoder weights (since `add_classifier=False, add_detector=False`). It will be reused (its *encoder* weights) as a warm start in `04_Train_MTL_Joint.ipynb`.